In [1]:
print("Connect")

Connect


In [ ]:
import torch
gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
  print('Not connected to a GPU. Go to Runtime > Change runtime type and select T4 GPU.')
else:
  print(gpu_info)

Tue Jul 28 10:29:29 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   64C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pip install -U transformers trl peft datasets accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 40.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 889.0/889.0 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 10.4 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.13.1
    Uninstalling transformers-5.13.1:
      Successfully uninstalled transformers-5.13.1


In [10]:
# scripts/download_base_model.py
import os
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
save_directory = "models/base"

print(f"Starting download of '{model_id}' from Hugging Face...")

# Ensure output directory exists
os.makedirs(save_directory, exist_ok=True)

# 1. Download and save the tokenizer files flat to models/base/
print("\nDownloading tokenizer files...")
tokenizer = AutoTokenizer.from_pretrained(model_id)

print(f"Saving tokenizer files directly to: {save_directory}")
tokenizer.save_pretrained(save_directory)

# 2. Download and save the model weights flat to models/base/
print("\nDownloading model weights (approx. 3.1 GB)...")
# We load on CPU to keep GPU memory free for your actual training runs
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,  # Baseline FP16 storage type
    device_map="cpu"            # Load on CPU to avoid using VRAM
)

print(f"Saving model weight shards directly to: {save_directory}...")
model.save_pretrained(save_directory)

print("\n--- Download and Saving Complete ---")
print(f"Your unquantized baseline weights are stored cleanly inside: '{save_directory}/'")

Starting download of 'Qwen/Qwen2.5-Coder-1.5B-Instruct' from Hugging Face...



config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Saving model weight shards directly to: models/base...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


--- Download and Saving Complete ---
Your unquantized baseline weights are stored cleanly inside: 'models/base/'


In [ ]:
import os

# Ensure the configs/ directory exists
os.makedirs("configs", exist_ok=True)

# 1. Define lora.yaml content (completely flat to avoid YAML indentation errors)
lora_yaml_content = """r: 16
lora_alpha: 32
lora_dropout: 0.05
bias: "none"
task_type: "CAUSAL_LM"
target_modules:
  - "q_proj"
  - "k_proj"
  - "v_proj"
  - "o_proj"
  - "gate_proj"
  - "up_proj"
  - "down_proj"
"""

# 2. Define train.yaml content
train_yaml_content = """model_id: "models/base"                 # Pointing directly to your flat base directory
output_dir: "models/checkpoints"        # Directory for intermediate checkpoints
train_file: "train.jsonl"
val_file: "validation.jsonl"

# Training Hyperparameters
learning_rate: 0.0005
per_device_train_batch_size: 2
gradient_accumulation_steps: 4
num_train_epochs: 5
max_length: 4096
logging_steps: 5
optim: "paged_adamw_8bit"
save_strategy: "epoch"                     # Avoids flooding drive with 3GB files during training
report_to: "none"                       # Disables third-party API logging during testing
"""

# Write lora.yaml
with open("configs/lora.yaml", "w", encoding="utf-8") as f:
    f.write(lora_yaml_content.strip())

# Write train.yaml
with open("configs/train.yaml", "w", encoding="utf-8") as f:
    f.write(train_yaml_content.strip())

print("Success! Created 'configs/lora.yaml' and 'configs/train.yaml' programmatically.")

Success! Created 'configs/lora.yaml' and 'configs/train.yaml' programmatically.


In [ ]:
import json
import os
import re

def sanitize_and_fix_jsonl(file_path):
    """
    Parses conversational structures by matching headers, extracts raw string fields,
    normalizes unescaped quotes/newlines, and overwrites the file with
    perfectly formatted, single-line JSONL rows.
    """
    if not os.path.exists(file_path):
        print(f"Error: File not found: {file_path}")
        return

    print(f"Repairing and sanitizing: {file_path}")
    try:
        with open(file_path, "r", encoding="utf-8") as f:
            full_text = f.read()

        # Split the file into separate conversation blocks
        blocks = re.split(r'\{\s*"\s*messages\s*"\s*:\s*\[', full_text)

        repaired_blocks = []
        header_pattern = r'\{\s*"\s*role\s*"\s*:\s*"\s*(system|user|assistant)\s*"\s*,\s*"\s*content\s*"\s*:\s*"'

        # The first split item contains leading file wrappers/whitespace, which we skip
        for block in blocks[1:]:
            block = block.strip()
            if not block:
                continue

            # Find all message headers inside this conversation block
            matches = list(re.finditer(header_pattern, block))
            if not matches:
                continue

            messages = []
            for i in range(len(matches)):
                role = matches[i].group(1)
                start_idx = matches[i].end()

                # Determine where the raw content segment ends
                if i < len(matches) - 1:
                    end_idx = matches[i + 1].start()
                    raw_segment = block[start_idx:end_idx].rstrip()
                    # Trim the trailing structural `"},` or `"} ,`
                    raw_segment = re.sub(r'"\s*\}\s*,\s*$', '', raw_segment)
                else:
                    # For the last message, read to the end of the block
                    raw_segment = block[start_idx:].rstrip()
                    # Trim trailing structural `"}]}` or similar conversation endings
                    raw_segment = re.sub(r'"\s*\}\s*\]\s*\}?\s*$', '', raw_segment)

                # Standardize the raw text segment
                # 1. Undo any partial escaping to prevent double-escaping (e.g. \" -> " and \\ -> \)
                decoded_text = raw_segment.replace('\\"', '"').replace('\\\\', '\\')

                messages.append({
                    "role": role,
                    "content": decoded_text
                })

            # Reconstruct as a perfectly standardized single-line JSON object
            reconstructed = {"messages": messages}
            repaired_blocks.append(json.dumps(reconstructed, ensure_ascii=False))

        # Overwrite the original file with the repaired lines
        with open(file_path, "w", encoding="utf-8") as out_f:
            for line in repaired_blocks:
                out_f.write(line + "\n")

        print(f"Successfully repaired '{file_path}'! Recoded {len(repaired_blocks)} valid entries.")
    except Exception as e:
        print(f"Failed to fix '{file_path}': {e}")
        raise e

sanitize_and_fix_jsonl("train.jsonl")

Repairing and sanitizing: train.jsonl
Successfully repaired 'train.jsonl'! Recoded 50 valid entries.


In [ ]:
# scripts/train.py
import os
import yaml
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTConfig, SFTTrainer

# 1. Load Local Configurations
print("Loading YAML configurations...")
with open("configs/train.yaml", "r") as f:
    train_config = yaml.safe_load(f)
with open("configs/lora.yaml", "r") as f:
    lora_config_dict = yaml.safe_load(f)

# Ensure checkpoints output folder exists
os.makedirs(train_config["output_dir"], exist_ok=True)

# 2. Configure 4-bit Quantization (QLoRA)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    bnb_4bit_use_double_quant=True
)

# 3. Load Local Model & Tokenizer
print(f"Loading local base model from '{train_config['model_id']}'...")
tokenizer = AutoTokenizer.from_pretrained(train_config["model_id"])
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"  # Required for training stability

model = AutoModelForCausalLM.from_pretrained(
    train_config["model_id"],
    quantization_config=bnb_config,
    device_map="auto"
)

# 4. Apply PEFT & Prepare for 4-bit Training
print("Applying LoRA adapters...")
model = prepare_model_for_kbit_training(model)
peft_config = LoraConfig(**lora_config_dict)

# 5. Load Dataset Splits
print("Loading split datasets...")
dataset_files = {
    "train": train_config["train_file"],
    "validation": train_config["val_file"]
}
dataset = load_dataset("json", data_files=dataset_files)

# 6. Initialize Training Configurations
training_args = SFTConfig(
    output_dir=train_config["output_dir"],
    per_device_train_batch_size=train_config["per_device_train_batch_size"],
    gradient_accumulation_steps=train_config["gradient_accumulation_steps"],
    learning_rate=float(train_config["learning_rate"]),
    logging_steps=train_config["logging_steps"],
    max_length=train_config["max_length"],
    num_train_epochs=train_config["num_train_epochs"],
    optim=train_config["optim"],
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    save_strategy=train_config["save_strategy"],
    save_total_limit=2,           # Keeps only the latest 2 checkpoints to prevent disk full errors
    report_to=train_config["report_to"],
    eval_strategy="epoch",        # Evaluates validation loss at the end of each epoch
    logging_dir="./logs/tensorboard"
)

# 7. Initialize Trainer
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    peft_config=peft_config,
    processing_class=tokenizer,
    args=training_args,
)

# 8. Check for Existing Checkpoints to Auto-Resume Training
resume_checkpoint = None
if os.path.exists(train_config["output_dir"]):
    # Check if any folders inside start with "checkpoint-"
    checkpoints = [
        os.path.join(train_config["output_dir"], d)
        for d in os.listdir(train_config["output_dir"])
        if d.startswith("checkpoint-") and os.path.isdir(os.path.join(train_config["output_dir"], d))
    ]
    if checkpoints:
        # Sort checkpoints based on global steps to find the latest folder
        checkpoints.sort(key=lambda x: int(x.split("-")[-1]))
        resume_checkpoint = checkpoints[-1]

# 9. Start Fine-Tuning
print("\n--- Starting Fine-Tuning Execution ---")
if resume_checkpoint:
    print(f"Found active checkpoint. Resuming from: {resume_checkpoint}")
    trainer.train(resume_from_checkpoint=resume_checkpoint)
else:
    print("No checkpoints found. Starting a fresh training run...")
    trainer.train()

# 10. Save final adapter weights to models/final/
adapter_save_dir = "models/final"
os.makedirs(adapter_save_dir, exist_ok=True)
trainer.model.save_pretrained(adapter_save_dir)
tokenizer.save_pretrained(adapter_save_dir)

print(f"\nTraining completed! Adapter weights saved cleanly inside '{adapter_save_dir}/'")

Loading YAML configurations...
Loading local base model from 'models/base'...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Applying LoRA adapters...
Loading split datasets...


Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

...

Dropping fully masked examples from eval dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.



--- Starting Fine-Tuning Execution ---
No checkpoints found. Starting a fresh training run...


Epoch,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
1,0.493441,0.461649,0.475241,86792.000000,0.874420
2,0.342183,0.401881,0.363005,173584.000000,0.891981
3,0.275034,0.381772,0.313857,260376.000000,0.904241
4,0.172559,0.409312,0.247504,347168.000000,0.901922
5,0.144453,0.444076,0.221329,433960.000000,0.903247



Training completed! Adapter weights saved cleanly inside 'models/final/'


In [ ]:
import shutil
from google.colab import files

print("Compressing checkpoints folder into a single ZIP file...")
# Compress models/checkpoints into checkpoints.zip
shutil.make_archive("final_merged", "zip", "models/final_merged")

print("\nStarting download to your local machine (File size: ~100-300MB)...")
files.download("final.zip")

Compressing checkpoints folder into a single ZIP file...

Starting download to your local machine (File size: ~100-300MB)...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import drive
import os

print("Mounting Google Drive...")
drive.mount('/content/drive')

# Copy the already-zipped file directly to your Google Drive root
print("Copying final_merged.zip to Google Drive (takes ~10 seconds)...")
!cp final_merged.zip /content/drive/MyDrive/final_merged.zip

print("\nSuccess! The file is safely stored in your Google Drive as 'final_merged_model.zip'.")
print("You can now open Google Drive on your computer and download it comfortably.")

Mounting Google Drive...
Mounted at /content/drive
Copying final_merged.zip to Google Drive (takes ~10 seconds)...

Success! The file is safely stored in your Google Drive as 'final_merged_model.zip'.
You can now open Google Drive on your computer and download it comfortably.


In [7]:
import gdown

# Replace 'YOUR_FILE_ID' with the actual file ID from your shareable link
url = 'https://drive.google.com/uc?id=1k6h-ZXSTOcQvvkSkfVK9L8QnjEFVnCWk'

# Download the file
output = 'final_merged.zip'
gdown.download(url, output, quiet=False)

Downloading...
From (original): https://drive.google.com/uc?id=1k6h-ZXSTOcQvvkSkfVK9L8QnjEFVnCWk
From (redirected): https://drive.google.com/uc?id=1k6h-ZXSTOcQvvkSkfVK9L8QnjEFVnCWk&confirm=t&uuid=58d43f79-56a4-4865-a023-6025d0c9e1dd
To: /content/final_merged.zip
100%|██████████| 2.46G/2.46G [01:05<00:00, 37.4MB/s]


'final_merged.zip'

In [ ]:
# scripts/merge_lora.py
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

base_model_path = "models/base"
adapter_path = "models/final"
save_path = "models/final_merged"

# 1. Clear VRAM and trigger garbage collection to free memory
import gc
torch.cuda.empty_cache()
gc.collect()

print("Checking hardware acceleration...")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Determine optimal dtype for weight merge
dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print(f"Using precision: {dtype}")

# 2. Load Base Tokenizer
print("Loading base tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(base_model_path)

# 3. Load Unquantized Base Model in 16-bit
print(f"Loading unquantized base model from '{base_model_path}'...")
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_path,
    torch_dtype=dtype,
    device_map=device
)

# 4. Attach the Trained Adapter to the Base Model
print(f"Loading adapter weights from '{adapter_path}'...")
peft_model = PeftModel.from_pretrained(base_model, adapter_path)

# 5. Mathematically Fuse Weights
print("Merging adapter weights with base model weights (weight fusion)...")
merged_model = peft_model.merge_and_unload()

# 6. Save Standalone Model
print(f"Saving standalone merged model directly to '{save_path}'...")
os.makedirs(save_path, exist_ok=True)
merged_model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

print("\n--- Weight Merge Complete ---")
print(f"Your custom standalone model weights are saved cleanly inside: '{save_path}/'")

Checking hardware acceleration...
Using device: cuda
Using precision: torch.bfloat16
Loading base tokenizer...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading unquantized base model from 'models/base'...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Loading adapter weights from 'models/final'...
Merging adapter weights with base model weights (weight fusion)...
Saving standalone merged model directly to 'models/final_merged'...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


--- Weight Merge Complete ---
Your custom standalone model weights are saved cleanly inside: 'models/final_merged/'


In [ ]:
!pip install torchao==0.16.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 84.6 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [8]:
!unzip final_merged -d model

Archive:  final_merged.zip
  inflating: model/tokenizer.json    
  inflating: model/model.safetensors  
  inflating: model/tokenizer_config.json  
  inflating: model/chat_template.jinja  
  inflating: model/config.json       
  inflating: model/generation_config.json  


In [1]:
# scripts/run_local_dual.py
import os
import sys
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# Paths to your models
tuned_model_path = "model"
base_model_path = "models/base"

# Ensure directories exist
for path, label in [(tuned_model_path, "Fine-tuned"), (base_model_path, "Base")]:
    if not os.path.exists(path) or not os.listdir(path):
        print(f"Error: {label} model directory '{path}' is empty or not found.")
        sys.exit(1)

# 1. Hardware Detection
print("Checking local hardware acceleration...")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Determine optimal data type based on hardware
if device == "cuda":
    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
else:
    dtype = torch.float32

# 2. Load Tokenizers and Models
print(f"\nLoading Base Model from '{base_model_path}'...")
tokenizer_base = AutoTokenizer.from_pretrained(base_model_path)
model_base = AutoModelForCausalLM.from_pretrained(base_model_path, torch_dtype=dtype).to(device)

print(f"Loading Fine-Tuned Model from '{tuned_model_path}'...")
tokenizer_tuned = AutoTokenizer.from_pretrained(tuned_model_path)
model_tuned = AutoModelForCausalLM.from_pretrained(tuned_model_path, torch_dtype=dtype).to(device)

print("\nBoth models are loaded and ready in memory.")

# 3. Define the specialized DevStudio system prompt
system_prompt = (
    "You are DevStudio-1.5B, an in-editor coding assistant developed by DevStudio AI. "
    "You are a highly specialized master of modern single-file HTML and Tailwind CSS designs. "
)

# 4. Interactive Terminal Loop
print("\n" + "="*60)
print("       DevStudio-1.5B Dual-Model Comparison Playground")
print("="*60)
print("Type your layout prompt and press Enter to compare outputs.")
print("Type 'exit' or 'quit' to close.")

while True:
    try:
        user_prompt = input("\nEnter prompt: ").strip()
        if not user_prompt:
            continue
        if user_prompt.lower() in ["exit", "quit"]:
            print("Exiting playground...")
            break

        # Structure inputs into the ChatML schema
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ]

        # --- GENERATION 1: BASE MODEL ---
        print("\n[1/2] Generating layout with Base Model...")
        prompt_base = tokenizer_base.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs_base = tokenizer_base(prompt_base, return_tensors="pt").to(model_base.device)

        with torch.no_grad():
            outputs_base = model_base.generate(
                **inputs_base,
                max_new_tokens=4086,   # Capped at 1024 for faster interactive responses
                temperature=0.2,
                do_sample=True,
                eos_token_id=tokenizer_base.eos_token_id
            )
        generated_ids_base = outputs_base[0][inputs_base["input_ids"].shape[1]:]
        response_base = tokenizer_base.decode(generated_ids_base, skip_special_tokens=True)

        # --- GENERATION 2: FINE-TUNED MODEL ---
        print("[2/2] Generating layout with Fine-Tuned Model...")
        prompt_tuned = tokenizer_tuned.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs_tuned = tokenizer_tuned(prompt_tuned, return_tensors="pt").to(model_tuned.device)

        with torch.no_grad():
            outputs_tuned = model_tuned.generate(
                **inputs_tuned,
                max_new_tokens=4086,   # Capped at 1024 for faster interactive responses
                temperature=0.2,
                do_sample=True,
                eos_token_id=tokenizer_tuned.eos_token_id
            )
        generated_ids_tuned = outputs_tuned[0][inputs_tuned["input_ids"].shape[1]:]
        response_tuned = tokenizer_tuned.decode(generated_ids_tuned, skip_special_tokens=True)

        # Post-process ONLY the fine-tuned model output to fix literal escapes
        response_tuned = response_tuned.replace("\\n", "\n")
        response_tuned = response_tuned.replace('\\"', '"')

        # --- DISPLAY SIDE-BY-SIDE / STACKED OUTPUTS ---
        print("\n" + "="*80)
        print(" [A] BASE MODEL OUTPUT:")
        print("="*80)
        print(response_base.strip())
        print("="*80)

        print("\n" + "="*80)
        print(" [B] FINE-TUNED MODEL (DEVSTUDIO-1.5B) OUTPUT:")
        print("="*80)
        print(response_tuned.strip())
        print("="*80)

    except KeyboardInterrupt:
        print("\nExiting...")
        break

Checking local hardware acceleration...
Using device: cuda

Loading Base Model from 'models/base'...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Loading Fine-Tuned Model from 'model'...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]


Both models are loaded and ready in memory.

       DevStudio-1.5B Dual-Model Comparison Playground
Type your layout prompt and press Enter to compare outputs.
Type 'exit' or 'quit' to close.

Enter prompt: modal overlay with clean input fields and buttons

[1/2] Generating layout with Base Model...
[2/2] Generating layout with Fine-Tuned Model...

 [A] BASE MODEL OUTPUT:
Sure! Below is a simple example of how you can create a modal overlay with clean input fields and buttons using HTML and Tailwind CSS.

```html
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Modal Overlay</title>
    <link href="https://cdn.jsdelivr.net/npm/tailwindcss@2.29.3/dist/tailwind.min.css" rel="stylesheet">
    <style>
        .modal-overlay {
            position: fixed;
            top: 0;
            left: 0;
            width: 100%;
            height: 100%;
            background-color: rgba(0, 0, 0